In [ ]:
# ============================================================
#  Inner Thinking Transformer (ITT) – GPT-2 scale
#  Architecture : ITT-GPT2 (Corrected Identity Path & Variance)
#  Training     : pure causal LM on BabyLM-2026-Strict-Small
# ============================================================

import os
import math
import time
import json
import random
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
import tiktoken

# ─────────────────────────────────────────────────────────────
# 0.  GLOBAL HYPERPARAMETERS
# ─────────────────────────────────────────────────────────────
VOCAB_SIZE        = 50257
BLOCK_SIZE        = 512
BATCH_SIZE        = 16
# FIX #1: Increased Gradient Accumulation to simulate a ~131k token batch size
GRAD_ACCUM_STEPS  = 16 
EPOCHS            = 10
LEARNING_RATE     = 1e-4
LR_MIN            = LEARNING_RATE * 0.1
WARMUP_STEPS      = 111 #no of steps in 1st epoch
WEIGHT_DECAY      = 0.1
GRAD_CLIP         = 1.0

BLIMP_FAST_PATH   = "fast_eval/blimp_fast"
SUPP_FAST_PATH    = "fast_eval/supplement_fast"


# ─────────────────────────────────────────────────────────────
# 1.  MODEL CONFIG 
# ─────────────────────────────────────────────────────────────

@dataclass
class GPTConfig:
    block_size: int = BLOCK_SIZE
    vocab_size: int = VOCAB_SIZE
    n_layer:    int = 12
    n_head:     int = 12
    n_embd:     int = 768


# ─────────────────────────────────────────────────────────────
# 2.  ARCHITECTURE 
# ─────────────────────────────────────────────────────────────

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        assert config.n_embd % config.n_head == 0

        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.RN_FLAGS_SCALE_DOWN = 1

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        head_dim = C // self.n_head
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y


class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu   = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.RN_FLAGS_SCALE_DOWN = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))


class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class ITTBlock(nn.Module):
    """
    Inner Thinking Transformer layer (§3 of the ITT paper).
      1. Residual Thinking Connection (RTC) – Eq. 4 / 7
      2. Adaptive Token Routing  (ATR)       – Eq. 5 / 6
      3. Thinking Step Encoding  (φ)
    """

    def __init__(self, config, max_thinking_steps: int = 4, selection_ratio: float = 0.7):
        super().__init__()
        self.config = config
        self.T   = max_thinking_steps
        self.rho = selection_ratio

        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

        self.routers = nn.ModuleList([
            nn.Linear(config.n_embd, 1, bias=False)
            for _ in range(self.T + 1)
        ])

        self.thinking_step_embeddings = nn.Parameter(
            torch.ones(self.T + 1, config.n_embd)
        )
        self.alpha = nn.Parameter(torch.ones(self.T + 1))

        # FIX #2: Zero-init the deeper thinking steps so the model starts stable
        # This prevents variance explosion at initialization.
        with torch.no_grad():
            self.thinking_step_embeddings[1:].zero_()
            self.alpha[1:].zero_()

    def _f(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

    def forward(self, x):
        B, T_seq, C = x.size()

        y_0           = self._f(x)
        phi_0         = self.thinking_step_embeddings[0].view(1, 1, C)
        y_accumulated = y_0 * phi_0
        y_prev        = y_0

        k = max(1, int(self.rho * T_seq))

        for t in range(1, self.T + 1):
            router_logits  = self.routers[t](y_prev).squeeze(-1)
            w_t            = torch.sigmoid(router_logits)

            _, topk_indices = torch.topk(w_t, k, dim=-1, largest=True, sorted=False)
            selection_mask  = torch.zeros(B, T_seq, dtype=torch.bool, device=x.device)
            selection_mask.scatter_(1, topk_indices, True)

            y_full       = self._f(y_prev)
            w_t_expand   = w_t.unsqueeze(-1)
            mask_expand  = selection_mask.unsqueeze(-1)

            # FIX #3: Extract the pure residual (delta) to prevent scaling the identity path
            y_delta = y_full - y_prev

            # Scale ONLY the new information (y_delta), then add it back to y_prev
            y_t_prime = torch.where(
                mask_expand,
                y_prev + (self.alpha[t] * w_t_expand * y_delta),
                y_prev
            )

            step_contribution = torch.where(
                mask_expand,
                y_t_prime,
                torch.zeros_like(y_prev)
            )
            phi_t         = self.thinking_step_embeddings[t].view(1, 1, C)
            y_accumulated = y_accumulated + step_contribution * phi_t
            y_prev        = y_t_prime

        return y_accumulated


class GPT(nn.Module):

    def __init__(self, config: GPTConfig,
                 max_thinking_steps: int = 4,
                 selection_ratio:    float = 0.7):
        super().__init__()
        self.config = config

        blocks = []
        for layer_id in range(config.n_layer):
            if layer_id % 2 == 1:
                blocks.append(ITTBlock(config, max_thinking_steps, selection_ratio))
            else:
                blocks.append(Block(config))

        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),
            wpe  = nn.Embedding(config.block_size, config.n_embd),
            h    = nn.ModuleList(blocks),
            ln_f = nn.LayerNorm(config.n_embd),
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)
        total = sum(p.numel() for p in self.parameters())
        print(f"[ITT-GPT2] Total parameters: {total / 1e6:.2f}M")

    def _init_weights(self, module):
        std = 0.02
        if hasattr(module, 'RN_FLAGS_SCALE_DOWN'):
            std *= (2 * self.config.n_layer) ** -0.5
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor,
                targets: torch.Tensor = None,
                bidirectional: bool = False):   # kept for eval-helper compatibility
        B, T = idx.size()
        assert T <= self.config.block_size, \
            f"Sequence length {T} exceeds block_size {self.config.block_size}"

        pos     = torch.arange(0, T, dtype=torch.long, device=idx.device)
        x       = self.transformer.wte(idx) + self.transformer.wpe(pos)

        for block in self.transformer.h:
            x = block(x)

        x      = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.config.vocab_size),
                targets.view(-1)
            )

        return logits, loss

    def configure_optimizers(self, weight_decay: float, learning_rate: float, device_type: str):
        param_dict     = {n: p for n, p in self.named_parameters() if p.requires_grad}
        decay_params   = [p for p in param_dict.values() if p.dim() >= 2]
        nodecay_params = [p for p in param_dict.values() if p.dim() < 2]
        groups = [
            {'params': decay_params,   'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0},
        ]
        num_decay   = sum(p.numel() for p in decay_params)
        num_nodecay = sum(p.numel() for p in nodecay_params)
        print(f"[Optimizer] decayed tensors:    {len(decay_params):4d}  | {num_decay:,} params")
        print(f"[Optimizer] non-decayed tensors:{len(nodecay_params):4d}  | {num_nodecay:,} params")
        fused_ok  = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_ok and ('cuda' in device_type)
        print(f"[Optimizer] fused AdamW: {use_fused}")
        return torch.optim.AdamW(groups, lr=learning_rate,
                                 betas=(0.9, 0.95), eps=1e-8, fused=use_fused)


# ─────────────────────────────────────────────────────────────
# 3.  DATA LOADER
# ─────────────────────────────────────────────────────────────

class DataLoaderLite:

    def __init__(self, B: int, T: int, texts: list, name: str = ""):
        self.B, self.T = B, T
        enc         = tiktoken.get_encoding("gpt2")
        self.eos_id = 50256

        print(f"[DataLoader:{name}] Tokenising {len(texts)} documents with tiktoken …")
        all_ids = []
        for t in texts:
            if t.strip():
                all_ids.extend(enc.encode(t, allowed_special={'<|endoftext|>'}))
                all_ids.append(self.eos_id)

        self.tokens     = torch.tensor(all_ids, dtype=torch.long)
        self.chunk_size = B * T
        self.n_chunks   = (len(self.tokens) - 1) // self.chunk_size
        self.indices    = list(range(self.n_chunks))
        self.pos        = 0
        self._shuffle()

        print(f"[DataLoader:{name}] Total tokens: {len(self.tokens):,} | "
              f"Iterations per epoch: {self.n_chunks:,}")

    def _shuffle(self):
        random.shuffle(self.indices)
        self.pos = 0

    def steps_per_epoch(self) -> int:
        return self.n_chunks

    def next_batch(self):
        if self.pos >= len(self.indices):
            self._shuffle()
        chunk_idx  = self.indices[self.pos]
        self.pos  += 1
        start      = chunk_idx * self.chunk_size
        temp       = self.tokens[start : start + self.chunk_size + 1]
        x = temp[:-1].view(self.B, self.T)
        y = temp[1: ].view(self.B, self.T)
        return x, y


# ─────────────────────────────────────────────────────────────
# 4.  LEARNING RATE SCHEDULER
# ─────────────────────────────────────────────────────────────

def get_lr(it: int, total_steps: int) -> float:
    if it < WARMUP_STEPS:
        return LEARNING_RATE * (it + 1) / WARMUP_STEPS
    if it >= total_steps:
        return LR_MIN
    decay_ratio = (it - WARMUP_STEPS) / (total_steps - WARMUP_STEPS)
    coeff       = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return LR_MIN + coeff * (LEARNING_RATE - LR_MIN)


# ─────────────────────────────────────────────────────────────
# 5.  ITT rho warm-up  (Algorithm 1 getCapacity)
# ─────────────────────────────────────────────────────────────

def get_selection_ratio(global_step: int, target_rho: float = 0.7) -> float:
    if global_step < WARMUP_STEPS:
        return target_rho * (global_step / WARMUP_STEPS)
    return target_rho


# ─────────────────────────────────────────────────────────────
# 6.  EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────

@torch.inference_mode()
def sentence_log_prob(sentence: str, model: GPT, enc,
                      device, autocast_ctx,
                      bidirectional: bool = False) -> float:
    ids = enc.encode(sentence, allowed_special={'<|endoftext|>'})
    if len(ids) <= 1:
        return -1e9
    ids = ids[:model.config.block_size]
    inp = torch.tensor([ids], dtype=torch.long, device=device)
    with autocast_ctx:
        logits, _ = model(inp, targets=None, bidirectional=bidirectional)
    shift_logits = logits[0, :-1, :]
    shift_labels = inp[0, 1:]
    lp = F.log_softmax(shift_logits, dim=-1)
    return lp[torch.arange(len(shift_labels)), shift_labels].sum().item()


def eval_blimp_folder(folder: str, model: GPT, enc,
                      device, autocast_ctx,
                      tag: str = "BLiMP") -> float:
    if not os.path.exists(folder):
        print(f"  [WARN] '{folder}' not found — skipping {tag}.")
        return 0.0
    total, correct = 0, 0
    for fn in sorted(os.listdir(folder)):
        if not fn.endswith('.jsonl'):
            continue
        with open(os.path.join(folder, fn), encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                sg   = item.get('sentence_good')
                sb   = item.get('sentence_bad')
                if sg and sb:
                    lp_g = sentence_log_prob(sg, model, enc, device, autocast_ctx)
                    lp_b = sentence_log_prob(sb, model, enc, device, autocast_ctx)
                    correct += int(lp_g > lp_b)
                    total   += 1
    acc = 100.0 * correct / total if total > 0 else 0.0
    print(f"  [{tag}] {correct}/{total} = {acc:.2f}%")
    return acc


def run_mid_epoch_eval(model: GPT, enc, device, autocast_ctx,
                       epoch: int, half: int):
    model.eval()
    label = f"Epoch {epoch+1}  " + ("— FIRST HALF ↓" if half == 0 else "— SECOND HALF ↓")
    print(f"\n{'='*65}\n  MID-EPOCH ZERO-SHOT EVAL  {label}\n{'='*65}")
    b_acc = 0.00 #skipping this evaln bechmark it takes lot of time 
    s_acc = eval_blimp_folder(SUPP_FAST_PATH,  model, enc, device, autocast_ctx, tag="Supplement-fast")
    print(f"  ➜  BLiMP-fast: {b_acc:.2f}%   Supplement-fast: {s_acc:.2f}%\n{'='*65}\n")
    model.train()
    return b_acc, s_acc


@torch.inference_mode()
def evaluate_val_loss(model: GPT, val_loader: DataLoaderLite,
                      device, autocast_ctx,
                      eval_iters: int = 20) -> float:
    model.eval()
    old_pos     = val_loader.pos
    old_indices = list(val_loader.indices)

    total = 0.0
    for _ in range(eval_iters):
        x, y = val_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with autocast_ctx:
            _, loss = model(x, y)
        total += loss.item()

    val_loader.indices = old_indices
    val_loader.pos     = old_pos
    model.train()
    return total / eval_iters


# ─────────────────────────────────────────────────────────────
# 7.  MAIN
# ─────────────────────────────────────────────────────────────

def main():
    torch.manual_seed(42)
    random.seed(42)

    device      = 'cuda' if torch.cuda.is_available() else 'cpu'
    device_type = 'cuda' if 'cuda' in device else 'cpu'
    amp_dtype   = torch.bfloat16
    autocast_ctx = torch.autocast(device_type=device_type, dtype=amp_dtype,
                                  enabled=(device_type == 'cuda'))
    if device_type == 'cuda':
        torch.set_float32_matmul_precision('high')
        torch.cuda.manual_seed(42)

    print(f"[Device] Using: {device}")

    enc = tiktoken.get_encoding("gpt2")

    print("\n[Data] Loading BabyLM-2026-Strict-Small …")
    ds       = load_dataset("BabyLM-community/BabyLM-2026-Strict-Small")
    all_text = list(ds['train']['text'])

    split       = int(len(all_text) * 0.95)
    train_texts = all_text[:split]
    val_texts   = all_text[split:]

    train_loader = DataLoaderLite(BATCH_SIZE, BLOCK_SIZE, train_texts, name="train")
    val_loader   = DataLoaderLite(BATCH_SIZE, BLOCK_SIZE, val_texts,   name="val")

    # Due to increased GRAD_ACCUM_STEPS, total optimization steps are reduced
    # So we divide the data loader chunks by GRAD_ACCUM_STEPS to find the true steps per epoch
    chunks_per_epoch = train_loader.steps_per_epoch()
    steps_per_epoch  = chunks_per_epoch // GRAD_ACCUM_STEPS
    total_steps      = steps_per_epoch * EPOCHS
    half_step        = steps_per_epoch // 2

    print(f"[Training] optimizations/epoch={steps_per_epoch:,}  total_optimizations={total_steps:,}")

    cfg   = GPTConfig()
    model = GPT(cfg, max_thinking_steps=4, selection_ratio=0.7).to(device)

    try:
        model = torch.compile(model)
        print("[Model] torch.compile() succeeded")
    except Exception as e:
        print(f"[Model] torch.compile() skipped ({e})")

    optimizer = model.configure_optimizers(WEIGHT_DECAY, LEARNING_RATE, device_type)
    os.makedirs("checkpoints", exist_ok=True)

    model.train()
    global_step = 0
    best_val    = float('inf')

    for epoch in range(EPOCHS):
        train_loader._shuffle()
        first_half_done  = False
        second_half_done = False

        for step in range(steps_per_epoch):
            t0 = time.perf_counter()

            lr = get_lr(global_step, total_steps)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            current_rho = get_selection_ratio(global_step, target_rho=0.7)
            raw_model   = model._orig_mod if hasattr(model, '_orig_mod') else model
            for block in raw_model.transformer.h:
                if isinstance(block, ITTBlock):
                    block.rho = current_rho

            optimizer.zero_grad(set_to_none=True)
            loss_accum = 0.0

            for _ in range(GRAD_ACCUM_STEPS):
                x, y = train_loader.next_batch()
                x, y = x.to(device), y.to(device)
                with autocast_ctx:
                    _, loss = model(x, y)
                scaled      = loss / GRAD_ACCUM_STEPS
                loss_accum += scaled.item()
                scaled.backward()

            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

            if device_type == 'cuda':
                torch.cuda.synchronize()

            dt  = (time.perf_counter() - t0) * 1000
            tps = (BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM_STEPS) / (dt / 1000)

            val_str = ""
            if global_step % 50 == 0:
                vl = evaluate_val_loss(model, val_loader, device, autocast_ctx, eval_iters=20)
                val_str = f"  val={vl:.4f}"
                if vl < best_val:
                    best_val  = vl
                    save_model = model._orig_mod if hasattr(model, '_orig_mod') else model
                    torch.save({
                        'step':                 global_step,
                        'epoch':                epoch,
                        'model_state_dict':     save_model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'config':               cfg,
                        'val_loss':             vl,
                    }, "checkpoints/itt_gpt2_best.pt")

            print(
                f"[E{epoch+1:02d} {step+1:>5d}/{steps_per_epoch} G{global_step:>7d}] "
                f"train={loss_accum:.4f}{val_str}  norm={norm:.3f}  "
                f"lr={lr:.2e}  rho={current_rho:.2f}  dt={dt:6.1f}ms  tok/s={tps:,.0f}"
            )

            if (step + 1) == half_step and not first_half_done:
                first_half_done = True
                run_mid_epoch_eval(model, enc, device, autocast_ctx, epoch, half=0)

            if (step + 1) == steps_per_epoch and not second_half_done:
                second_half_done = True
                run_mid_epoch_eval(model, enc, device, autocast_ctx, epoch, half=1)

            global_step += 1

        save_model = model._orig_mod if hasattr(model, '_orig_mod') else model
        torch.save({
            'epoch':                epoch + 1,
            'step':                 global_step,
            'model_state_dict':     save_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config':               cfg,
        }, f"checkpoints/itt_gpt2_epoch{epoch+1:02d}.pt")


if __name__ == "__main__":
    main()